# 心思维｜已有 TorchCREPE 与 VocalParse 音高交叉验证

本 Notebook 只读取 Google Drive 中已有的 `vocal_pitch_raw.npz`，并使用内嵌的 25 秒 VocalParse 测试样本。**不访问 GitHub、不克隆仓库、不下载模型、不运行 CREPE/VocalParse 推理，也不需要 GPU。**

只需运行下面唯一的代码单元格。结果会保存到 `TorchCREPE_Test` 文件夹。

In [ ]:
from google.colab import drive
from pathlib import Path
import json, math, statistics
import numpy as np

# Drive 已挂载时出现 already mounted 属于正常提示
drive.mount('/content/drive')

# 本单元格完全自包含：不访问 GitHub、不克隆仓库、不下载模型
drive_root = Path('/content/drive/MyDrive/XinsW_Lead_Transcription/TorchCREPE_Test')
npz_path = drive_root / 'vocal_pitch_raw.npz'
track_json = drive_root / 'reference_pitch_track.json'
comparison_json = drive_root / 'vocalparse_pitch_comparison_30_55s.json'
if not npz_path.is_file():
    raise FileNotFoundError(f'Drive 中找不到已有音高文件：{npz_path}')

# 25 秒 VocalParse 固定样本（对应歌曲 30–55 秒片段）
parsed = {
  "bpm": 100,
  "words": [
    "SP",
    "如",
    "果",
    "世",
    "·",
    "界",
    "变",
    "·",
    "换",
    "太",
    "快",
    "SP",
    "当",
    "青",
    "·",
    "春",
    "SP",
    "慢",
    "慢",
    "走",
    "·",
    "远",
    "SP",
    "当",
    "容",
    "·",
    "颜",
    "不",
    "再",
    "如",
    "初",
    "·",
    "·",
    "见",
    "SP",
    "你",
    "不",
    "必",
    "问",
    "我",
    "SP",
    "爱",
    "·",
    "不",
    "·",
    "·",
    "爱",
    "SP",
    "也",
    "不"
  ],
  "pitches": [
    0,
    55,
    55,
    58,
    59,
    60,
    61,
    58,
    60,
    58,
    54,
    0,
    55,
    58,
    60,
    60,
    0,
    58,
    58,
    54,
    56,
    56,
    0,
    55,
    55,
    54,
    53,
    54,
    52,
    49,
    53,
    55,
    54,
    54,
    0,
    47,
    47,
    49,
    50,
    52,
    0,
    52,
    54,
    56,
    58,
    56,
    56,
    0,
    56,
    55
  ],
  "notes": [
    "NOTE_2",
    "NOTE_8",
    "NOTE_DOT_16",
    "NOTE_8",
    "NOTE_DOT_16",
    "NOTE_DOT_4",
    "NOTE_DOT_16",
    "NOTE_DOT_16",
    "NOTE_4",
    "NOTE_DOT_8",
    "NOTE_DOT_4",
    "NOTE_2",
    "NOTE_8",
    "NOTE_DOT_8",
    "NOTE_4",
    "NOTE_DOT_4",
    "NOTE_DOT_4",
    "NOTE_DOT_8",
    "NOTE_4",
    "NOTE_8",
    "NOTE_16",
    "NOTE_4",
    "NOTE_DOT_8",
    "NOTE_8",
    "NOTE_DOT_16",
    "NOTE_DOT_16",
    "NOTE_8",
    "NOTE_8",
    "NOTE_8",
    "NOTE_4",
    "NOTE_DOT_8",
    "NOTE_16",
    "NOTE_DOT_16",
    "NOTE_2",
    "NOTE_4",
    "NOTE_8",
    "NOTE_8",
    "NOTE_DOT_8",
    "NOTE_DOT_16",
    "NOTE_DOT_4",
    "NOTE_16",
    "NOTE_8",
    "NOTE_DOT_8",
    "NOTE_DOT_8",
    "NOTE_DOT_16",
    "NOTE_DOT_16",
    "NOTE_DOT_4",
    "NOTE_DOT_4",
    "NOTE_8",
    "NOTE_8"
  ]
}

def convert_existing_npz(path):
    with np.load(path, allow_pickle=False) as archive:
        required = {'pitch', 'periodicity'}
        missing = required.difference(archive.files)
        if missing:
            raise ValueError(f'NPZ 缺少数组 {sorted(missing)}；实际包含 {archive.files}')
        hz = np.asarray(archive['pitch']).reshape(-1).astype(np.float64)
        confidence = np.asarray(archive['periodicity']).reshape(-1).astype(np.float64)
    if len(hz) != len(confidence):
        raise ValueError('pitch 与 periodicity 数组长度不一致')
    if not np.isfinite(hz).all() or (hz < 0).any():
        raise ValueError('pitch 必须为有限非负 Hz 数值')
    if not np.isfinite(confidence).all() or ((confidence < 0) | (confidence > 1)).any():
        raise ValueError('periodicity 必须为 0 到 1 之间的有限数值')
    frames = [{'time_sec': i * 0.01, 'frequency_hz': float(hz[i]), 'confidence': float(confidence[i])} for i in range(len(hz))]
    result = {'source_file': path.name, 'frame_count': len(frames), 'timing': {'source': 'confirmed_hop_seconds', 'hop_seconds': 0.01}, 'pitch_representation': 'frequency_hz', 'frames': frames}
    track_json.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return result

DURATION_BEATS = {'NOTE_1':4.0,'NOTE_DOT_1':6.0,'NOTE_2':2.0,'NOTE_DOT_2':3.0,'NOTE_4':1.0,'NOTE_DOT_4':1.5,'NOTE_8':0.5,'NOTE_DOT_8':0.75,'NOTE_16':0.25,'NOTE_DOT_16':0.375,'NOTE_32':0.125,'NOTE_DOT_32':0.1875}
def make_events(data):
    if not (len(data['words']) == len(data['pitches']) == len(data['notes'])):
        raise ValueError('VocalParse words/pitches/notes 数量不一致')
    bpm = float(data['bpm'])
    if not math.isfinite(bpm) or bpm <= 0: raise ValueError('BPM 无效')
    events, beat = [], 0.0
    for i, (word, pitch, token) in enumerate(zip(data['words'], data['pitches'], data['notes'])):
        if token not in DURATION_BEATS: raise ValueError(f'不支持的时值：{token}')
        if isinstance(pitch, bool) or int(pitch) != float(pitch) or not 0 <= int(pitch) <= 127: raise ValueError(f'无效 MIDI 音高：{pitch}')
        duration = DURATION_BEATS[token]; pitch = int(pitch); rest = (word == 'SP' or pitch == 0)
        events.append({'event_id': f'vp-{i+1:05d}', 'raw_word': str(word), 'lyric': None if rest or word == '·' else str(word), 'pitch_midi': None if rest else pitch, 'duration_token': token, 'duration_beats': duration, 'start_beat': beat, 'end_beat': beat + duration, 'is_rest': rest, 'is_lyric_continuation': word == '·'})
        beat += duration
    return events

def compare_events(events, bpm, frames, confidence_threshold=0.5, central_fraction=0.6, offset=30.0):
    parsed_frames = []
    for f in frames:
        hz = float(f['frequency_hz']); midi = None if hz == 0 else 69.0 + 12.0 * math.log2(hz / 440.0)
        parsed_frames.append((float(f['time_sec']), midi, float(f['confidence'])))
    parsed_frames.sort(key=lambda x: x[0])
    trim = (1.0 - central_fraction) / 2.0; seconds_per_beat = 60.0 / float(bpm)
    rows, errors, rest_ratios = [], [], []
    for i, event in enumerate(events):
        start = event['start_beat'] * seconds_per_beat; end = (event['start_beat'] + event['duration_beats']) * seconds_per_beat
        a = start + (end-start)*trim + offset; b = end - (end-start)*trim + offset
        selected = [(p,c) for t,p,c in parsed_frames if a <= t <= b]
        voiced = [(p,c) for p,c in selected if p is not None and c >= confidence_threshold]
        row = {'event_index':i,'raw_word':event['raw_word'],'start_sec':start,'end_sec':end,'reference_start_sec':start+offset,'reference_end_sec':end+offset,'is_rest':event['is_rest'],'frame_count':len(selected),'voiced_frame_count':len(voiced)}
        if event['is_rest']:
            ratio = len(voiced)/len(selected) if selected else None; row['reference_voiced_ratio'] = ratio
            if ratio is not None: rest_ratios.append(ratio)
        else:
            row['vocalparse_pitch_midi'] = event['pitch_midi']
            if voiced:
                ref = float(statistics.median(p for p,c in voiced)); err = float(event['pitch_midi']) - ref
                row.update({'reference_median_pitch_midi':ref,'pitch_error_semitones':err,'reference_confidence_mean':statistics.mean(c for p,c in voiced),'is_octave_mismatch':abs(abs(err)-12.0)<=1.0})
                errors.append((err,row['is_octave_mismatch']))
            else: row.update({'reference_median_pitch_midi':None,'pitch_error_semitones':None,'is_octave_mismatch':None})
        rows.append(row)
    abs_errors = [abs(e) for e,_ in errors]; note_count = sum(not e['is_rest'] for e in events)
    summary = {'note_count':note_count,'compared_note_count':len(errors),'uncompared_note_count':note_count-len(errors),'median_absolute_error_semitones':float(statistics.median(abs_errors)) if abs_errors else None,'within_one_semitone_rate':sum(e<=1.0 for e in abs_errors)/len(abs_errors) if abs_errors else None,'octave_mismatch_count':sum(flag for _,flag in errors),'rest_count':sum(bool(e['is_rest']) for e in events),'compared_rest_count':len(rest_ratios),'mean_rest_voiced_ratio':float(statistics.mean(rest_ratios)) if rest_ratios else None,'confidence_threshold':confidence_threshold,'central_fraction':central_fraction,'reference_time_offset_sec':offset,'comparison_type':'model_to_model_not_ground_truth'}
    return {'summary':summary,'events':rows}

converted = convert_existing_npz(npz_path)
events = make_events(parsed)
report = compare_events(events, parsed['bpm'], converted['frames'], confidence_threshold=0.5, central_fraction=0.6, offset=30.0)
comparison_json.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('参考音高帧数：', converted['frame_count'])
print('采样间隔：0.01 秒；比较片段：歌曲 30–55 秒')
print('\n===== 交叉验证汇总 =====')
print(json.dumps(report['summary'], ensure_ascii=False, indent=2))
print('\n===== 音高偏差最大的前 15 个音符 =====')
rows = []
for event, row in zip(events, report['events']):
    if not event['is_rest'] and row['pitch_error_semitones'] is not None:
        rows.append({'字':event['raw_word'],'VocalParse MIDI':row['vocalparse_pitch_midi'],'TorchCREPE MIDI中位数':round(row['reference_median_pitch_midi'],2),'误差(半音)':round(row['pitch_error_semitones'],2),'疑似八度错误':row['is_octave_mismatch'],'有效帧数':row['voiced_frame_count']})
rows.sort(key=lambda item:abs(item['误差(半音)']),reverse=True)
for row in rows[:15]: print(row)
print('\n完整比较报告：', comparison_json)
print('说明：periodicity 阈值 0.5 仅用于筛选，不代表人工真值；本报告比较的是两个模型输出。')
